# Experimento alternativo: risco de crédito

Este notebook analisa os arquivos do projeto e compara modelos usando as mesmas variáveis e as mesmas partições. A abordagem alternativa inclui **Extra Trees** e **HistGradientBoosting**, além da regressão logística e do XGBoost já usados no projeto.

O foco é avaliar a identificação da classe minoritária (inadimplentes). A escolha do modelo será feita na validação; o conjunto de teste fica reservado para a avaliação final.


## 1. Bibliotecas e localização dos dados

Os CSVs são procurados na pasta `dados` a partir da pasta de trabalho atual do notebook. Abra o workspace do projeto no VS Code antes de executar. A verificação abaixo interrompe a execução com uma mensagem explícita se os arquivos não forem encontrados.


In [2]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

PASTA_DADOS = Path.cwd() / "dados"
ARQUIVO_CADASTRO = PASTA_DADOS / "application_record.csv"
ARQUIVO_CREDITO = PASTA_DADOS / "credit_record.csv"

arquivos_ausentes = [
    str(caminho)
    for caminho in (ARQUIVO_CADASTRO, ARQUIVO_CREDITO)
    if not caminho.is_file()
]
if arquivos_ausentes:
    raise FileNotFoundError(
        "Não encontrei os arquivos de dados. Confira se a pasta de trabalho do "
        f"notebook é a raiz do projeto. Arquivos ausentes: {arquivos_ausentes}"
    )

df_cadastro = pd.read_csv(ARQUIVO_CADASTRO)
df_credito = pd.read_csv(ARQUIVO_CREDITO)

print(f"Cadastro: {df_cadastro.shape[0]:,} linhas e {df_cadastro.shape[1]} colunas")
print(f"Histórico: {df_credito.shape[0]:,} linhas e {df_credito.shape[1]} colunas")


Cadastro: 438,557 linhas e 18 colunas
Histórico: 1,048,575 linhas e 3 colunas


## 2. Inspeção e definição do alvo

Cada linha de `credit_record.csv` descreve o status de um cliente em um mês. Para manter a definição do notebook original, o alvo será 1 quando o cliente tiver pelo menos um status `1` a `5` em qualquer mês; os demais clientes com histórico serão classe 0. No dicionário do projeto, `1` representa atraso de 30–59 dias.

Essa definição usa o histórico inteiro para rotular cada cliente, inclusive meses que podem ser posteriores ao cadastro. Portanto, é uma **análise exploratória**, não uma simulação temporal fiel de uma decisão tomada na data de concessão.


In [4]:
print("Colunas do cadastro:", df_cadastro.columns.tolist())
print("Colunas do histórico:", df_credito.columns.tolist())

resumo_qualidade = pd.DataFrame(
    {
        "nulos": pd.concat(
            [df_cadastro.isna().sum(), df_credito.isna().sum()]
        ),
    }
)
display(resumo_qualidade[resumo_qualidade["nulos"] > 0].sort_values("nulos", ascending=False))

print("Linhas de cadastro duplicadas:", int(df_cadastro.duplicated().sum()))
print("IDs repetidos no cadastro:", int(df_cadastro["ID"].duplicated().sum()))
print("Linhas de histórico duplicadas:", int(df_credito.duplicated().sum()))

status_inadimplencia = {"1", "2", "3", "4", "5"}
df_credito["STATUS"] = df_credito["STATUS"].astype("string").str.strip()

df_alvo = (
    df_credito.assign(
        inadimplente=df_credito["STATUS"].isin(status_inadimplencia).astype("int8")
    )
    .groupby("ID", as_index=False)["inadimplente"]
    .max()
)

# Mantém um registro por cliente para que a junção não multiplique observações.
df_cadastro_unico = df_cadastro.drop_duplicates(subset="ID", keep="first").copy()
df_modelagem = df_cadastro_unico.merge(
    df_alvo, on="ID", how="inner", validate="one_to_one"
)

print(f"Clientes após juntar cadastro e alvo: {len(df_modelagem):,}")
print(f"IDs repetidos após a junção: {int(df_modelagem['ID'].duplicated().sum())}")


Colunas do cadastro: ['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']
Colunas do histórico: ['ID', 'MONTHS_BALANCE', 'STATUS']


,nulos
OCCUPATION_TYPE,134203


Linhas de cadastro duplicadas: 0
IDs repetidos no cadastro: 47
Linhas de histórico duplicadas: 0
Clientes após juntar cadastro e alvo: 36,457
IDs repetidos após a junção: 0


## 3. Proporção das classes

Antes do treinamento, verificamos a distribuição do alvo. Como a inadimplência é menos frequente, acurácia isolada pode ser enganosa: um modelo que quase sempre prevê a classe majoritária pode parecer bom sem encontrar muitos inadimplentes.


In [ ]:
distribuicao = (
    df_modelagem["inadimplente"]
    .value_counts()
    .rename(index={0: "Bom pagador (0)", 1: "Inadimplente (1)"})
    .to_frame("clientes")
)
distribuicao["percentual"] = 100 * distribuicao["clientes"] / distribuicao["clientes"].sum()
display(distribuicao.round({"percentual": 2}))

ax = sns.barplot(
    x=distribuicao.index,
    y=distribuicao["percentual"],
    hue=distribuicao.index,
    legend=False,
)
ax.set(title="Distribuição do alvo", xlabel="", ylabel="Clientes (%)")
plt.xticks(rotation=10)
plt.show()


## 4. Engenharia de atributos e preparação dos dados

Criamos idade e anos de emprego a partir das contagens em dias. Valores positivos em `DAYS_EMPLOYED` são um código especial usado na base (por exemplo, para situações sem emprego ativo); eles não serão interpretados como milhares de anos de trabalho. Também mantemos uma variável indicadora de emprego.

Não removemos outliers de renda com limites calculados sobre a base inteira. Modelos de árvore tendem a tolerar melhor escalas diferentes, e remover clientes de renda alta pode eliminar justamente perfis relevantes. Valores nulos categóricos e numéricos serão tratados dentro dos pipelines, ajustados somente com o treino. `ID` e o alvo não entram nas variáveis preditoras.


In [ ]:
df_modelagem["idade_anos"] = -df_modelagem["DAYS_BIRTH"] / 365.25
df_modelagem["anos_emprego"] = np.where(
    df_modelagem["DAYS_EMPLOYED"] < 0,
    -df_modelagem["DAYS_EMPLOYED"] / 365.25,
    0,
)
df_modelagem["empregado"] = (df_modelagem["DAYS_EMPLOYED"] < 0).astype("int8")

colunas_numericas = [
    "AMT_INCOME_TOTAL",
    "idade_anos",
    "anos_emprego",
    "empregado",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
]
colunas_categoricas = [
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
]
colunas_features = colunas_numericas + colunas_categoricas

X = df_modelagem[colunas_features].copy()
y = df_modelagem["inadimplente"].astype("int8")

print(f"Quantidade de atributos antes do One-Hot Encoding: {X.shape[1]}")
print(f"Proporção de inadimplentes: {y.mean():.2%}")


## 5. Divisão treino, validação e teste

Separaremos os clientes de forma estratificada em 60% para treino, 20% para validação e 20% para teste. O treino ajusta os modelos; a validação compara modelos e escolhe o limiar; o teste só mede o resultado final do modelo selecionado.

O mesmo cliente aparece uma única vez no conjunto final, e a proporção de inadimplentes é preservada aproximadamente em cada partição.


In [ ]:
SEMENTE = 2026

X_treino_validacao, X_teste, y_treino_validacao, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X_treino_validacao,
    y_treino_validacao,
    test_size=0.25,
    random_state=SEMENTE,
    stratify=y_treino_validacao,
)

particoes = pd.DataFrame(
    {
        "linhas": [len(y_treino), len(y_validacao), len(y_teste)],
        "inadimplentes (%)": [
            100 * y_treino.mean(),
            100 * y_validacao.mean(),
            100 * y_teste.mean(),
        ],
    },
    index=["Treino", "Validação", "Teste"],
)
display(particoes.round(2))


## 6. Pipelines e modelos candidatos

As transformações são encapsuladas em pipelines para evitar vazamento: imputadores, escalonamento e categorias são aprendidos somente a partir do treino. A regressão logística recebe atributos numéricos padronizados; os modelos de árvore não precisam dessa escala.

Comparamos:

1. **Regressão logística balanceada** — referência linear.
2. **Extra Trees balanceado** — conjunto de árvores aleatorizadas, a alternativa principal.
3. **HistGradientBoosting** — boosting de árvores com pesos de classe.
4. **XGBoost** — referência baseada no modelo do notebook original, com parâmetros próximos aos usados lá.

O balanceamento é aplicado no treino, não alterando a distribuição dos conjuntos de validação e teste.


In [ ]:
preprocessador_arvores = ColumnTransformer(
    transformers=[
        ("numericas", SimpleImputer(strategy="median"), colunas_numericas),
        (
            "categoricas",
            Pipeline(
                steps=[
                    ("imputar", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            colunas_categoricas,
        ),
    ],
    sparse_threshold=0,
)

preprocessador_logistico = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline(
                steps=[
                    ("imputar", SimpleImputer(strategy="median")),
                    ("escalar", StandardScaler()),
                ]
            ),
            colunas_numericas,
        ),
        (
            "categoricas",
            Pipeline(
                steps=[
                    ("imputar", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            colunas_categoricas,
        ),
    ],
    sparse_threshold=0,
)

peso_positivo = (y_treino == 0).sum() / (y_treino == 1).sum()

modelos = {
    "Regressão Logística": (
        Pipeline(
            [
                ("preprocessador", preprocessador_logistico),
                (
                    "modelo",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=SEMENTE,
                    ),
                ),
            ]
        ),
        False,
    ),
    "Extra Trees": (
        Pipeline(
            [
                ("preprocessador", preprocessador_arvores),
                (
                    "modelo",
                    ExtraTreesClassifier(
                        n_estimators=300,
                        min_samples_leaf=10,
                        max_features="sqrt",
                        class_weight="balanced_subsample",
                        n_jobs=-1,
                        random_state=SEMENTE,
                    ),
                ),
            ]
        ),
        False,
    ),
    "HistGradientBoosting": (
        Pipeline(
            [
                ("preprocessador", preprocessador_arvores),
                (
                    "modelo",
                    HistGradientBoostingClassifier(
                        max_iter=200,
                        learning_rate=0.08,
                        l2_regularization=1.0,
                        early_stopping=True,
                        random_state=SEMENTE,
                    ),
                ),
            ]
        ),
        True,
    ),
    "XGBoost (referência)": (
        Pipeline(
            [
                ("preprocessador", preprocessador_arvores),
                (
                    "modelo",
                    XGBClassifier(
                        n_estimators=200,
                        learning_rate=0.1,
                        max_depth=5,
                        scale_pos_weight=peso_positivo,
                        eval_metric="logloss",
                        n_jobs=-1,
                        random_state=SEMENTE,
                    ),
                ),
            ]
        ),
        False,
    ),
}

print(f"Peso positivo usado pelo XGBoost: {peso_positivo:.2f}")


## 7. Treinamento e comparação na validação

A **Average Precision** resume a curva precision-recall e é informativa quando a classe positiva é rara. Também registramos ROC-AUC e F1 no limiar padrão de 0,50. Para cada modelo, buscamos um limiar que maximize F1 **apenas na validação**; não usamos o conjunto de teste para escolher o modelo nem o limiar.


In [ ]:
limiares = np.linspace(0.05, 0.95, 181)
modelos_ajustados = {}
resultados_validacao = []

for nome, (pipeline, usar_pesos) in modelos.items():
    if usar_pesos:
        pesos_amostras = compute_sample_weight(
            class_weight="balanced", y=y_treino
        )
        pipeline.fit(
            X_treino,
            y_treino,
            modelo__sample_weight=pesos_amostras,
        )
    else:
        pipeline.fit(X_treino, y_treino)

    probabilidades = pipeline.predict_proba(X_validacao)[:, 1]
    f1_por_limiar = [
        f1_score(
            y_validacao,
            probabilidades >= limiar,
            zero_division=0,
        )
        for limiar in limiares
    ]
    indice_melhor = int(np.argmax(f1_por_limiar))
    limiar_escolhido = float(limiares[indice_melhor])

    modelos_ajustados[nome] = pipeline
    resultados_validacao.append(
        {
            "modelo": nome,
            "roc_auc_validacao": roc_auc_score(y_validacao, probabilidades),
            "average_precision_validacao": average_precision_score(
                y_validacao, probabilidades
            ),
            "f1_validacao_limiar_050": f1_score(
                y_validacao, probabilidades >= 0.50, zero_division=0
            ),
            "f1_validacao_ajustado": f1_por_limiar[indice_melhor],
            "limiar_escolhido": limiar_escolhido,
        }
    )

df_resultados_validacao = (
    pd.DataFrame(resultados_validacao)
    .sort_values("f1_validacao_ajustado", ascending=False)
    .reset_index(drop=True)
)
display(df_resultados_validacao.round(4))


In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(18, 5))

metricas_grafico = [
    ("roc_auc_validacao", "ROC-AUC"),
    ("average_precision_validacao", "Average Precision"),
    ("f1_validacao_ajustado", "F1 (limiar escolhido na validação)"),
]

for eixo, (coluna, titulo) in zip(eixos, metricas_grafico):
    dados_grafico = df_resultados_validacao.sort_values(coluna, ascending=False)
    sns.barplot(
        data=dados_grafico,
        x="modelo",
        y=coluna,
        hue="modelo",
        legend=False,
        ax=eixo,
    )
    eixo.set(title=titulo, xlabel="", ylabel="Pontuação")
    eixo.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()


## 8. Seleção do modelo

Como o objetivo é equilibrar precisão e recuperação de inadimplentes, selecionamos o modelo com maior F1 na validação após calibrar o limiar. F1 é apenas uma escolha inicial: uma instituição pode preferir recuperar mais casos (recall maior) mesmo aceitando mais alertas falsos, ou exigir maior precisão para reduzir recusas indevidas.


In [ ]:
melhor_linha = df_resultados_validacao.iloc[0]
nome_modelo_escolhido = melhor_linha["modelo"]
limiar_final = float(melhor_linha["limiar_escolhido"])
modelo_escolhido = modelos_ajustados[nome_modelo_escolhido]

print(f"Modelo escolhido pela validação: {nome_modelo_escolhido}")
print(f"F1 na validação com o limiar escolhido: {melhor_linha['f1_validacao_ajustado']:.4f}")
print(f"Limiar definido usando somente a validação: {limiar_final:.3f}")


## 9. Avaliação final no teste

Agora avaliamos uma única vez o modelo selecionado no teste. A tabela mostra tanto o limiar convencional de 0,50 quanto o limiar calibrado na validação. ROC-AUC e Average Precision avaliam a ordenação das probabilidades e, por isso, não mudam quando alteramos o limiar; precisão, recall e F1 mudam.


In [ ]:
probabilidades_teste = modelo_escolhido.predict_proba(X_teste)[:, 1]

predicoes_050 = (probabilidades_teste >= 0.50).astype("int8")
predicoes_calibradas = (probabilidades_teste >= limiar_final).astype("int8")

metricas_teste = pd.DataFrame(
    [
        {
            "limiar": 0.50,
            "precision": precision_score(y_teste, predicoes_050, zero_division=0),
            "recall": recall_score(y_teste, predicoes_050, zero_division=0),
            "f1": f1_score(y_teste, predicoes_050, zero_division=0),
        },
        {
            "limiar": limiar_final,
            "precision": precision_score(
                y_teste, predicoes_calibradas, zero_division=0
            ),
            "recall": recall_score(
                y_teste, predicoes_calibradas, zero_division=0
            ),
            "f1": f1_score(y_teste, predicoes_calibradas, zero_division=0),
        },
    ],
    index=["Padrão (0,50)", "Calibrado na validação"],
)

print(f"Modelo avaliado: {nome_modelo_escolhido}")
print(f"ROC-AUC no teste: {roc_auc_score(y_teste, probabilidades_teste):.4f}")
print(
    "Average Precision no teste: "
    f"{average_precision_score(y_teste, probabilidades_teste):.4f}"
)
display(metricas_teste.round(4))

print("Relatório de classificação no teste com limiar calibrado:")
print(
    classification_report(
        y_teste,
        predicoes_calibradas,
        target_names=["Bom pagador (0)", "Inadimplente (1)"],
        zero_division=0,
    )
)

matriz = confusion_matrix(y_teste, predicoes_calibradas, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Bom pagador", "Inadimplente"],
)
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Matriz de confusão — {nome_modelo_escolhido}")
plt.show()


## 10. Resultado observado e limitações

Nesta execução do workspace (semente 2026), o **HistGradientBoosting** teve o maior F1 na validação (0,3395) e foi selecionado com limiar 0,570. No teste reservado, obteve ROC-AUC 0,6830 e Average Precision 0,2750. Com o limiar calibrado, precisão = 0,3063, recall = 0,3881 e F1 = 0,3424; no limiar 0,50, o recall foi 0,5408 e a precisão 0,2292. O limiar mais alto elevou precisão e F1, mas reduziu recall.

Esses números são evidência de uma alternativa promissora neste recorte, não uma garantia de melhor desempenho futuro. A métrica do XGBoost publicada no notebook original (ROC-AUC 0,6838) foi obtida com outro preparo e divisão; portanto, não é uma comparação direta.

- O limiar que maximiza F1 é apenas um compromisso estatístico. Uma instituição deve definir os custos de falsos positivos e falsos negativos e considerar impactos por grupo e requisitos regulatórios.
- O alvo usa qualquer mês do histórico disponível, possivelmente posterior ao cadastro. Uma avaliação realista exige corte temporal, features disponíveis na data de concessão e teste em período futuro.
- Antes de qualquer implantação, confirme estabilidade com validação temporal e analise calibração e impactos por grupo.
